<img src='https://tools.actris-ares.eu/wp-content/uploads/2025/07/Actris-150x150.png' align='right' style='max-height: 90px;'>

## Explore ACTRIS Facilities

The goal of this notebook is to explore the ACTRIS Facilities and ACTRIS National Facilities through the ACTRIS Metadata REST API — a machine-to-machine way of reaching the metadata, without going through the web portal.

Browse the facilities on a map, filter them by country, and see which ones hold the ACTRIS label.

Let's get started!

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

### Using ACTRIS metadata catalog REST API

ACTRIS metadata catalog REST API: https://prod-actris-md2.nilu.no/index.html

The ACTRIS Rest API uses the ACTRIS vocabulary for several of the search criteria, the vocabulary can be found here: https://vocabulary.actris.nilu.no/skosmos/actris_vocab/en/

### Import libraries

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "Explore ACTRIS Facilities"
NOTEBOOK_VERSION = "2.1"
NOTEBOOK_DATE = "2026-09-14"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

import importlib.util
import subprocess
import sys

# Only the packages that are actually missing are installed. Each VRE user has their own
# container, so a notebook that runs for a colleague can still fail here.
REQUIRED = {
    "pandas": "pandas",
    "requests": "requests",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    # Hidden dependency: plotly needs nbformat to render figures but does not declare it
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    print("--> Done." if result.returncode == 0 else result.stderr[-1500:])
else:
    print("--> All required packages are already installed.")

import json
import os

import pandas as pd
import requests

import ipywidgets as widgets
from IPython.display import display

import plotly
import plotly.express as px
import plotly.offline as pyo

print("plotly:", plotly.__version__)

In [ ]:
# To work in offline mode
pyo.init_notebook_mode()

### Facilities

In [ ]:
response = requests.get("https://prod-actris-md2.nilu.no/facilities") # get all facilities in metadata archive
archive = response.json()
df = pd.DataFrame(archive)

# To display all rows without truncation
#pd.set_option('display.max_rows', None)  # show every row instead of truncating the table
#display(df)
# Restore the default option (optional)
#pd.reset_option('display.max_rows')

In [ ]:
df['actris_national_facility'] = df['actris_national_facility'].astype(str).str.lower().map({'true': True, 'false': False})

fig = px.scatter_geo(
    df,
    lat="lat",
    lon="lon",
    color="actris_national_facility",            # Use the boolean column
    color_discrete_map={True: "red", False: "steelblue"},  # True in red
    hover_name="name",
    projection="natural earth",
    size_max=15,
    width=1000,
    height=500,
    labels={
        "actris_national_facility": "ACTRIS National Facility"
    }
)

# Layout: margins + legend title and consistent ordering
fig.update_layout(
    margin=dict(l=20, r=20, t=20, b=20),
    legend_title_text="ACTRIS National Facility",
    legend=dict(
        orientation="h",            
        yanchor="bottom",
        y=-0.15,                    
        xanchor="left",
        x=0
    ),
)

> ACTRIS [National Facilities](https://www.actris.eu/facilities/national-facilities) have
> officially passed the strict ACTRIS labelling process to meet pan-European quality standards,
> while non-national (or unlabelled) facilities do not yet formally hold this official status
> within the ACTRIS network.

In [ ]:
# --- Widget: country code selection (dropdown) ---
# Extract available country codes from the dataframe
country_codes = sorted(df['country_code'].dropna().astype(str).unique().tolist())

country_dd = widgets.Dropdown(
    options=country_codes,
    value=country_codes[0] if country_codes else None,
    description='Country:',
    disabled=False
)

# Checkbox for the "ACTRIS National Facility" filter
nf_checkbox = widgets.Checkbox(
    value=False,  # by default show all facilities (no NF filter)
    description='Show only ACTRIS National Facilities',
    indent=False
)

status = widgets.HTML()
out = widgets.Output()

def apply_filter(change=None):
    # base filter: country chosen from dropdown
    selected_country = country_dd.value
    if selected_country is None:
        with out:
            out.clear_output(wait=True)
            display(widgets.HTML("<b>No country selected.</b>"))
        return

    mask = (df['country_code'].astype(str) == str(selected_country))

    # optional filter: ACTRIS National Facility
    if nf_checkbox.value:
        mask = mask & (df['actris_national_facility'] == True)
        status.value = f"<b>Active filter:</b> {selected_country} + ACTRIS National Facilities"
    else:
        status.value = f"<b>Active filter:</b> {selected_country} (all facilities)"

    facilities_country = df[mask]

    # update output without duplicates
    with out:
        out.clear_output(wait=True)
        display(status, facilities_country)

# --- Wiring: first render ---
country_dd.observe(apply_filter, names='value')
nf_checkbox.observe(apply_filter, names='value')

# UI + initial render
display(widgets.HBox([country_dd, nf_checkbox]), out)
apply_filter()

<a id="export"></a>
## Keeping the results

Like the other ACTRIS in-situ notebooks of this VRE, this one **keeps nothing between runs**: the
facility list is read from the ACTRIS metadata API every time, and the map is an interactive Plotly
chart that lives only in this page. There is no output folder to empty and no export cell.

**To keep the map**, hover over it and use the **camera icon** in the toolbar that appears in the
top-right corner: it saves a PNG to your browser's download folder. The same toolbar has zoom and
pan, worth using on a map this dense.

**To keep the table instead**, run the cell below. It writes the facility list — names, codes,
coordinates and the National Facility flag — as a CSV next to this notebook, which you can then
download from the file browser on the left (right-click the file, *Download*).

Note that this is the one file this notebook can write, and only if you ask it to.

In [ ]:
from pathlib import Path

csv_path = Path("facilities.csv")
df.to_csv(csv_path, index=False)

print(f"Saved {len(df)} facilities to {csv_path.resolve()}")
print("Download it from the file browser on the left: right-click the file, then Download.")